In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
%config InlineBackend.figure_format = 'retina'

import torch
from torchvision.models import resnet50
import torchvision.transforms as T
torch.set_grad_enabled(False)

# COCO classes
CLASSES = [
    'N/A', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A',
    'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse',
    'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack',
    'umbrella', 'N/A', 'N/A', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis',
    'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove',
    'skateboard', 'surfboard', 'tennis racket', 'bottle', 'N/A', 'wine glass',
    'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich',
    'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake',
    'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table', 'N/A',
    'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard',
    'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'N/A',
    'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier',
    'toothbrush'
]


transform = T.Compose([
    T.Resize(800),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# For output bounding box post-processing
def box_cxcywh_to_xyxy(x):
    x_c, y_c, w, h = x.unbind(1)
    b = [(x_c - 0.5 * w), (y_c - 0.5 * h),
         (x_c + 0.5 * w), (y_c + 0.5 * h)]
    return torch.stack(b, dim=1)

def rescale_bboxes(out_bbox, size):
    img_w, img_h = size
    b = box_cxcywh_to_xyxy(out_bbox)
    # Create tensor on the same device as out_bbox to avoid device mismatch
    b = b * torch.tensor([img_w, img_h, img_w, img_h], dtype=torch.float32, device=out_bbox.device)
    return b

def detect(im, model, transform, device, active_heads=None, active_layers=None):
    # Mean-std normalize the input image (batch-size: 1)
    img = transform(im).unsqueeze(0)

    img = img.to(device)
    
    # Propagate through the model
    if active_heads is not None:
        outputs = model(img, effective_heads=active_heads)
    elif active_layers is not None:
        outputs = model(img, active_layers=active_layers)
    else:
        outputs = model(img)
        
    # Keep only predictions with 0.7+ confidence
    probas = outputs['pred_logits'].softmax(-1)[0, :, :-1]
    keep = probas.max(-1).values > 0.7

    # Convert boxes from [0; 1] to image scales
    bboxes_scaled = rescale_bboxes(outputs['pred_boxes'][0, keep], im.size)
    return probas[keep], bboxes_scaled

def predict(im, model, transform, device, active_heads=None, active_layers=None):
    # Mean-std normalize the input image (batch-size: 1)
    img = transform(im).unsqueeze(0)

    img = img.to(device)
    
    # Propagate through the model
    if active_heads is not None:
        outputs = model(img, effective_heads=active_heads)
    elif active_layers is not None:
        outputs = model(img, active_layers=active_layers)
    else:
        outputs = model(img)
        
    probas = outputs['pred_logits'].softmax(-1)[0, :, :-1]
    bboxes_scaled = rescale_bboxes(outputs['pred_boxes'][0], im.size)
    return probas, bboxes_scaled

In [ ]:
import sys
from pathlib import Path
import torch 
# Add project paths
project_root = Path(r'C:\workspace\ml\workspace\master\original')
sys.path.insert(0, str(project_root))

from models import build_model
from evaluation.args import HeadArgs

# Initialize custom DETR model with 4 heads
args = HeadArgs(number_of_heads=4)
model, criterion, postprocessors = build_model(args)

# Load checkpoint
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

checkpoint_path = project_root / 'results' / 'baseline' / '4head' / 'checkpoint.pth'
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint['model'])
    print(f"Loaded checkpoint from {checkpoint_path}")
else:
    print(f"Warning: Checkpoint not found at {checkpoint_path}")

model.eval()
print(f"Custom DETR model with {args.nheads} heads loaded on {device}")

In [ ]:
import sys
from pathlib import Path
import torch 
# Add project paths
project_root = Path(r'C:\workspace\ml\workspace\master\original')
sys.path.insert(0, str(project_root))

from models import build_model
from evaluation.args import HeadArgs

# Initialize custom DETR model with 4 heads
args = HeadArgs(number_of_heads=4)
model, criterion, postprocessors = build_model(args)

# Load checkpoint
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

checkpoint_path = project_root / 'results' / 'baseline' / '4head' / 'checkpoint.pth'
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint['model'])
    print(f"Loaded checkpoint from {checkpoint_path}")
else:
    print(f"Warning: Checkpoint not found at {checkpoint_path}")

model.eval()
print(f"Custom DETR model with {args.nheads} heads loaded on {device}")

In [ ]:
import sys
from pathlib import Path
import torch 
# Add project paths
project_root = Path(r'C:\workspace\ml\workspace\master\original')
sys.path.insert(0, str(project_root))

from models import build_model
from evaluation.args import HeadArgs

# Initialize custom DETR model with 4 heads
args = HeadArgs(number_of_heads=1)
model, criterion, postprocessors = build_model(args)

# Load checkpoint
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

checkpoint_path = project_root / 'results' / 'baseline' / '1head' / 'checkpoint.pth'
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint['model'])
    print(f"Loaded checkpoint from {checkpoint_path}")
else:
    print(f"Warning: Checkpoint not found at {checkpoint_path}")

model.eval()
print(f"Custom DETR model with {args.nheads} heads loaded on {device}")

In [ ]:
import sys
from pathlib import Path
import torch 
# Add project paths
project_root = Path(r'C:\workspace\ml\workspace\master\original')
sys.path.insert(0, str(project_root))

from sliced_models import build_model
from evaluation.args import HeadArgs

# Initialize custom DETR model with 4 heads
args = HeadArgs(number_of_heads=4)
model, criterion, postprocessors = build_model(args)

# Load checkpoint
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

checkpoint_path = project_root / 'results' / 'sliced' / '300epoch' / 'checkpoint.pth'
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint['model'])
    print(f"Loaded checkpoint from {checkpoint_path}")
else:
    print(f"Warning: Checkpoint not found at {checkpoint_path}")

model.eval()
print(f"Custom DETR model with {args.nheads} heads loaded on {device}")

In [ ]:
import sys
from pathlib import Path
import torch 
# Add project paths
project_root = Path(r'C:\workspace\ml\workspace\master\original')
sys.path.insert(0, str(project_root))

from layer_scaling import build_model
from evaluation.args import HeadArgs

# Initialize custom DETR model with 4 heads
args = HeadArgs(number_of_heads=4)
model, criterion, postprocessors = build_model(args)

# Load checkpoint
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

checkpoint_path = project_root / 'results' / 'layer_scaling' / '300epoch' / 'checkpoint.pth'
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint['model'])
    print(f"Loaded checkpoint from {checkpoint_path}")
else:
    print(f"Warning: Checkpoint not found at {checkpoint_path}")

model.eval()
print(f"Custom DETR model with {args.nheads} heads loaded on {device}")

In [ ]:
import random
import json
from pathlib import Path
from collections import Counter

# ── Setup COCO dataset ────────────────────────────────────────────────────────
coco_path = 'C:\\workspace\\ml\\data\\coco'
image_set = 'val'
root = Path(coco_path)
assert root.exists(), f'provided COCO path {root} does not exist'

mode = 'instances'
PATHS = {
    "train": (root / "train2017", root / "annotations" / f'{mode}_train2017.json'),
    "val": (root / "val2017", root / "annotations" / f'{mode}_val2017.json'),
}

img_folder, ann_file = PATHS[image_set]

with open(ann_file) as f:
    coco_data = json.load(f)

# ── Select image with selected classes ────────────────────────────────────────
# Find images that contain annotations with selected class indices
category_id_to_idx = {cat['id']: i for i, cat in enumerate(coco_data['categories'])}

valid_image_ids = set()
for ann in coco_data['annotations']:
    cat_idx = category_id_to_idx.get(ann['category_id'])
    if ann['category_id'] in [36]:  # person, car, truck, snowboard
        valid_image_ids.add(ann['image_id'])

if valid_image_ids:
    selected_image_id = random.choice(list(valid_image_ids))
    random_image_info = next(img for img in coco_data['images'] if img['id'] == selected_image_id)
    print(f"Found {len(valid_image_ids)} images with selected classes")
else:
    # Fallback to random image if no images found with selected classes
    print(f"No images found with selected classes, using random image")
    random_image_info = random.choice(coco_data['images'])

file_name = "000000402118.jpg"  # Replace with the desired file name
image_path = img_folder / file_name
im = Image.open(image_path).convert("RGB")

print(f"Selected image: {file_name}")
print(f"Image size: {im.size}")

# ── Run detection ─────────────────────────────────────────────────────────────
scores, boxes = detect(im, model, transform, device)
print(f"Detected {len(scores)} objects with confidence > 0.7")



In [ ]:
import random
import json
from pathlib import Path
from collections import Counter

# ── Setup COCO dataset ────────────────────────────────────────────────────────
coco_path = 'C:\\workspace\\ml\\data\\coco'
image_set = 'val'
root = Path(coco_path)
assert root.exists(), f'provided COCO path {root} does not exist'

mode = 'instances'
PATHS = {
    "train": (root / "train2017", root / "annotations" / f'{mode}_train2017.json'),
    "val": (root / "val2017", root / "annotations" / f'{mode}_val2017.json'),
}

img_folder, ann_file = PATHS[image_set]

with open(ann_file) as f:
    coco_data = json.load(f)

# ── Select image with selected classes ────────────────────────────────────────
# Find images that contain annotations with selected class indices
category_id_to_idx = {cat['id']: i for i, cat in enumerate(coco_data['categories'])}

valid_image_ids = set()
for ann in coco_data['annotations']:
    cat_idx = category_id_to_idx.get(ann['category_id'])
    if ann['category_id'] in [36]:  # person, car, truck, snowboard
        valid_image_ids.add(ann['image_id'])

if valid_image_ids:
    selected_image_id = random.choice(list(valid_image_ids))
    random_image_info = next(img for img in coco_data['images'] if img['id'] == selected_image_id)
    print(f"Found {len(valid_image_ids)} images with selected classes")
else:
    # Fallback to random image if no images found with selected classes
    print(f"No images found with selected classes, using random image")
    random_image_info = random.choice(coco_data['images'])

file_name = "000000402118.jpg"  # Replace with the desired file name
#file_name = random_image_info['file_name']
image_path = img_folder / file_name
im = Image.open(image_path).convert("RGB")

print(f"Selected image: {file_name}")
print(f"Image size: {im.size}")

# ── Run detection ─────────────────────────────────────────────────────────────
#scores, boxes = detect(im, model, transform, device, active_layers=1)



In [ ]:
scores = predict(im, model, transform, device, active_heads=1)

In [ ]:
# ── Visualize Top 10 Query Scores ────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-poster')

query_idx = 0  # Change this to visualize different queries
print(f"\nVisualizing scores for Query {query_idx}:")
# Convert scores to numpy array
if torch.is_tensor(scores):
    scores_flat = scores.cpu().numpy()[query_idx].flatten()
else:
    scores_flat = np.array(scores)[query_idx].flatten()

print("Box: ", boxes[query_idx].cpu().numpy())
# Get top 10 highest scores with their indices
top_10_indices = np.argsort(scores_flat)[-10:][::-1]
top_10_scores = scores_flat[top_10_indices]
top_10_classes = [CLASSES[idx] if idx < len(CLASSES) else f"Class {idx}" for idx in top_10_indices]

# Create figure with only top 10 bar chart
fig, ax = plt.subplots(figsize=(12, 8))
fig.suptitle(f'Top 10 Highest Class Scores - Query {query_idx}', fontsize=16, fontweight='bold')

bars = ax.barh(range(len(top_10_classes)), top_10_scores, color='blue', edgecolor='black', linewidth=1.5)
ax.set_yticks(range(len(top_10_classes)))
ax.set_yticklabels(top_10_classes, fontsize=12)
ax.set_xlabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Top 10 Highest Class Scores', fontsize=14, fontweight='bold', pad=20)
ax.grid(True, alpha=0.3, axis='x')
ax.invert_yaxis()

# Add value labels on bars
for i, (bar, score) in enumerate(zip(bars, top_10_scores)):
    ax.text(score + 0.01, i, f'{score:.4f}', va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nTop 10 Classes (out of {len(scores_flat)} total):")
for i, (cls_name, score) in enumerate(zip(top_10_classes, top_10_scores), 1):
    print(f"  {i:2d}. {cls_name:20s} - {score:.4f}")


# Encoder Attention Heatmap Visualization

Visualize attention maps from encoder layers by overlaying them on the original image as a heatmap.

In [ ]:
# Capture encoder features using hooks
encoder_features = {}

def create_hook(layer_idx):
    def hook_fn(module, input, output):
        encoder_features[f'layer_{layer_idx}'] = output
    return hook_fn

# Register hooks for each encoder layer
for i, layer in enumerate(model.transformer.encoder.layers):
    layer.register_forward_hook(create_hook(i))

# Prepare image and run inference to capture encoder features
if 'img_tensor' not in locals():
    # If img_tensor doesn't exist, load the image
    img_tensor = transform(im).unsqueeze(0).to(device)

# Run inference to populate encoder_features
with torch.no_grad():
    outputs = model(img_tensor)

print(f"Captured encoder features from {len(encoder_features)} layers")
print(f"Encoder features keys: {list(encoder_features.keys())}")


In [ ]:
import cv2
import numpy as np

# Postprocess encoder features to create attention maps
def postprocess_encoder_activations(features):
    """Convert encoder feature maps to heatmaps"""
    # features shape: [seq_len, batch, hidden_dim]
    output = torch.abs(features)
    output = torch.sum(output, dim=-1).squeeze()  # Sum across hidden dim
    output = output.cpu().numpy()
    output = cv2.resize(output, (224, 224))
    output /= output.max()
    output *= 255
    return 255 - output.astype('uint8')

# Apply heatmap to visualize attention
def apply_heatmap_encoder(weights, img_tensor):
    """Overlay heatmap on image"""
    # Convert image tensor to numpy
    img_np = img_tensor.cpu().numpy()
    if img_np.ndim == 4:
        img_np = img_np.squeeze(0)
    # Denormalize: reverse the normalization applied in transform
    img_np = img_np * np.array([0.229, 0.224, 0.225])[:, None, None] + np.array([0.485, 0.456, 0.406])[:, None, None]
    img_np = (img_np.transpose(1, 2, 0) * 255).astype('uint8')
    # Resize to match heatmap size
    img_np = cv2.resize(img_np, (224, 224))
    # Convert RGB to BGR for OpenCV
    img_np = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
    
    heatmap = cv2.applyColorMap(weights, cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(heatmap, 0.7, img_np, 0.3, 0)
    # Convert back to RGB for matplotlib
    overlay = cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)
    return overlay

print("Encoder heatmap functions defined")


In [ ]:
# Visualize encoder attention across multiple layers
if 'encoder_features' in locals():
    num_layers = len(encoder_features)
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    for idx, ax in enumerate(axes.flat):
        if idx < num_layers:
            encoder_feats = encoder_features[f'layer_{idx}']
            weights = postprocess_encoder_activations(encoder_feats)
            overlay = apply_heatmap_encoder(weights, img_tensor)
            ax.imshow(overlay)
            ax.set_title(f'Layer {idx}')
        ax.axis('off')
    
    plt.suptitle('Encoder Attention Heatmaps Across Layers', fontsize=16, y=1.00)
    plt.tight_layout()
    plt.show()
